# Lab 3.2: Data Preprocessing & CORINE Label Extraction

## ⏱️ Time Allocation
- **Part 1 (15 min):** Understanding CORINE Land Cover data
- **Part 2 (20 min):** Setting up the processing environment
- **Part 3 (30 min):** Processing Sentinel-2 tiles with CORINE labels
- **Part 4 (15 min):** Analyzing and visualizing extracted patches

## 🎯 Learning Objectives

### Core (Essential - Everyone Should Complete)
- ✅ Understand CORINE Land Cover classification system
- ✅ Align CORINE data with Sentinel-2 imagery
- ✅ Extract labeled training patches for ML models
- ✅ Analyze class distributions in training data

### Optional (For Early Finishers)
- 🔵 Process multiple S2 tiles in batch
- 🔵 Customize patch size and stride
- 🔵 Implement class balancing strategies

---

## Section 1: Introduction to CORINE Land Cover

### What is CORINE?
**CORINE** (Coordination of Information on the Environment) Land Cover is a European programme providing consistent land cover/land use information across Europe.

### Key Features:
- 🗺️ **Coverage:** All EU member states + some neighboring countries
- 📐 **Resolution:** 100m (raster version)
- 🏷️ **Classes:** 44 land cover classes in 3-level hierarchy
- 📅 **Updates:** 1990, 2000, 2006, 2012, 2018 (we use 2018)

### Class Hierarchy (simplified):
```
Level 1: Artificial surfaces, Agricultural areas, Forest, Wetlands, Water bodies
Level 2: Urban fabric, Industrial areas, Arable land, Forests, etc.
Level 3: 44 detailed classes (what we use)
```

### ⚠️ Important Note
CORINE does **NOT** cover Iceland! If you're working with Icelandic tiles, you'll need alternative land cover data (e.g., ESA WorldCover).

## Section 2: Environment Setup

### Import Required Libraries

In [1]:
import os
import sys
import json
import subprocess
from pathlib import Path
from glob import glob
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
from osgeo import gdal

# Configure GDAL
gdal.UseExceptions()
os.environ['GDAL_CACHEMAX'] = '2048'

# Set matplotlib config directory to avoid permission issues
os.environ['MPLCONFIGDIR'] = f"/tmp/matplotlib_{os.environ.get('USER', 'user')}"

print("✓ Libraries imported successfully")
print(f"  GDAL version: {gdal.__version__}")
print(f"  NumPy version: {np.__version__}")

Matplotlib created a temporary cache directory at /tmp/matplotlib-b4ra4rlr because the default path (/p/home/jusers/hashim1/jureca/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


ModuleNotFoundError: No module named 'osgeo'

### Define CORINE Class Mapping

The CORINE raster uses simplified codes 1-44 (not the full CLC codes like 111, 112, etc.).

In [ ]:
# CORINE class descriptions (simplified codes 1-44 used in raster)
CORINE_CLASSES = {
    # Artificial surfaces (1-11)
    1: "Continuous urban fabric",
    2: "Discontinuous urban fabric",
    3: "Industrial or commercial units",
    4: "Road and rail networks",
    5: "Port areas",
    6: "Airports",
    7: "Mineral extraction sites",
    8: "Dump sites",
    9: "Construction sites",
    10: "Green urban areas",
    11: "Sport and leisure facilities",
    
    # Agricultural areas (12-22)
    12: "Non-irrigated arable land",
    13: "Permanently irrigated land",
    14: "Rice fields",
    15: "Vineyards",
    16: "Fruit trees and berry plantations",
    17: "Olive groves",
    18: "Pastures",
    19: "Annual crops with permanent crops",
    20: "Complex cultivation patterns",
    21: "Agriculture with natural vegetation",
    22: "Agro-forestry areas",
    
    # Forest and semi-natural areas (23-34)
    23: "Broad-leaved forest",
    24: "Coniferous forest",
    25: "Mixed forest",
    26: "Natural grasslands",
    27: "Moors and heathland",
    28: "Sclerophyllous vegetation",
    29: "Transitional woodland-shrub",
    30: "Beaches, dunes, sands",
    31: "Bare rocks",
    32: "Sparsely vegetated areas",
    33: "Burnt areas",
    34: "Glaciers and perpetual snow",
    
    # Wetlands (35-39)
    35: "Inland marshes",
    36: "Peat bogs",
    37: "Salt marshes",
    38: "Salines",
    39: "Intertidal flats",
    
    # Water bodies (40-44)
    40: "Water courses",
    41: "Water bodies",
    42: "Coastal lagoons",
    43: "Estuaries",
    44: "Sea and ocean",
    
    48: "No data"
}

# Color mapping for visualization
CORINE_COLORS = {
    1: '#E6004D', 2: '#FF0000', 3: '#CC4DF2', 4: '#CC0000', 5: '#E6CCCC',
    6: '#E6CCE6', 7: '#A600CC', 8: '#A64DCC', 9: '#FF4DFF', 10: '#FFA6FF',
    11: '#FFE6FF', 12: '#FFFFA8', 13: '#FFFF00', 14: '#E6E600', 15: '#E68000',
    16: '#F2A64D', 17: '#E6A600', 18: '#E6E64D', 19: '#FFE6A6', 20: '#FFE64D',
    21: '#E6CC4D', 22: '#F2CCA6', 23: '#80FF00', 24: '#00A600', 25: '#4DFF00',
    26: '#CCF24D', 27: '#A6FF80', 28: '#A6E64D', 29: '#A6F200', 30: '#E6E6E6',
    31: '#CCCCCC', 32: '#CCFFCC', 33: '#000000', 34: '#A6E6CC', 35: '#A6A6FF',
    36: '#4D4DFF', 37: '#CCCCFF', 38: '#E6E6FF', 39: '#A6A6E6', 40: '#00CCF2',
    41: '#80F2E6', 42: '#00FFA6', 43: '#A6FFE6', 44: '#E6F2FF'
}

print(f"✓ Defined {len(CORINE_CLASSES)} CORINE land cover classes")

## Section 3: Configure Processing Parameters

### 🔧 Set Your Paths and Options

**Edit the cell below to configure:**
1. Which Sentinel-2 tiles to process
2. Patch extraction parameters
3. Output directory

In [ ]:
# ============================================================
# CONFIGURATION - Edit these values!
# ============================================================

# Get username for default paths
USER = os.environ.get('USER', 'user')

# --- Path Configuration ---
# Directory containing your downloaded Sentinel-2 .SAFE folders
S2_DATA_DIR = f"/p/scratch/training2600/{USER}/data"

# CORINE Land Cover raster (shared location)
CORINE_PATH = "/p/scratch/training2600/CORINE/u2018_clc2018_v2020_20u1_raster100m/DATA/U2018_CLC2018_V2020_20u1.tif"

# Output directory for training data
OUTPUT_DIR = f"/p/scratch/training2600/{USER}/training_data"

# --- Tile Selection ---
# Options:
#   'all'  - Process all .SAFE directories found in S2_DATA_DIR
#   'list' - Process only tiles listed in TILE_LIST below
TILE_SELECTION = 'all'

# If TILE_SELECTION = 'list', specify which tiles to process:
TILE_LIST = [
    # Add your specific tile names here, e.g.:
    # "S2A_MSIL2A_20181019T102031_N0500_R065_T33UUP_20230813T105225.SAFE",
    # "S2B_MSIL2A_20210615T100559_N0500_R022_T33UVP_20230516T143510.SAFE",
]

# --- Patch Extraction Parameters ---
PATCH_SIZE = 3       # Size in pixels (3 = 30m x 30m at 10m resolution)
STRIDE = None        # Stride for extraction (None = same as PATCH_SIZE, no overlap)
MAX_PATCHES = 50000  # Maximum patches per tile

# --- Processing Options ---
SKIP_EXISTING = True  # Skip tiles that already have output files
CREATE_VISUALIZATIONS = True  # Create sample patch visualizations

# ============================================================
print("Configuration:")
print(f"  S2 Data Directory: {S2_DATA_DIR}")
print(f"  CORINE Path: {CORINE_PATH}")
print(f"  Output Directory: {OUTPUT_DIR}")
print(f"  Tile Selection: {TILE_SELECTION}")
print(f"  Patch Size: {PATCH_SIZE}x{PATCH_SIZE} ({PATCH_SIZE * 10}m x {PATCH_SIZE * 10}m)")
print(f"  Max Patches per Tile: {MAX_PATCHES:,}")

### Discover Available Tiles

In [ ]:
def find_s2_tiles(data_dir, selection='all', tile_list=None):
    """
    Find Sentinel-2 .SAFE directories to process.
    
    Parameters:
    -----------
    data_dir : str
        Directory containing .SAFE folders
    selection : str
        'all' to find all tiles, 'list' to use tile_list
    tile_list : list
        List of specific tile names to process
    
    Returns:
    --------
    list : List of Path objects for .SAFE directories
    """
    data_path = Path(data_dir)
    
    if not data_path.exists():
        print(f"❌ Data directory not found: {data_dir}")
        return []
    
    if selection == 'all':
        # Find all .SAFE directories
        tiles = sorted(data_path.glob("*.SAFE"))
    else:
        # Use specific list
        tiles = []
        for tile_name in (tile_list or []):
            tile_path = data_path / tile_name
            if tile_path.exists():
                tiles.append(tile_path)
            else:
                print(f"⚠ Tile not found: {tile_name}")
    
    return tiles

# Find tiles
available_tiles = find_s2_tiles(S2_DATA_DIR, TILE_SELECTION, TILE_LIST)

print(f"\n📂 Found {len(available_tiles)} Sentinel-2 tile(s) to process:")
for i, tile in enumerate(available_tiles, 1):
    print(f"  {i}. {tile.name}")

if len(available_tiles) == 0:
    print("\n⚠ No tiles found! Check your S2_DATA_DIR path.")

### Verify CORINE Data

In [ ]:
# Check CORINE file exists and get basic info
corine_path = Path(CORINE_PATH)

if not corine_path.exists():
    print(f"❌ CORINE file not found: {CORINE_PATH}")
    print("\nPlease check the path or download CORINE data.")
else:
    ds = gdal.Open(str(corine_path), gdal.GA_ReadOnly)
    print("✓ CORINE Land Cover 2018")
    print(f"  File: {corine_path.name}")
    print(f"  Size: {ds.RasterXSize} x {ds.RasterYSize} pixels")
    print(f"  Resolution: 100m x 100m")
    print(f"  Projection: EPSG:3035 (ETRS89-LAEA Europe)")
    ds = None

## Section 4: Processing Functions

These functions handle the core processing steps:
1. **Stack S2 bands** - Combine B02, B03, B04, B08 into a single GeoTIFF
2. **Align CORINE** - Reproject CORINE to match S2 tile geometry
3. **Extract patches** - Create training samples with labels

In [ ]:
def stack_s2_bands(safe_path, output_path):
    """
    Stack Sentinel-2 bands (B02, B03, B04, B08) into a single GeoTIFF.
    
    Parameters:
    -----------
    safe_path : Path
        Path to .SAFE directory
    output_path : Path
        Output path for stacked GeoTIFF
    
    Returns:
    --------
    bool : Success status
    """
    bands = ['B02', 'B03', 'B04', 'B08']  # Blue, Green, Red, NIR
    resolution = '10m'
    
    try:
        # Find band files
        band_paths = {}
        for band_name in bands:
            pattern = f"**/R{resolution}/*_{band_name}_{resolution}.jp2"
            matches = list(safe_path.glob(pattern))
            if matches:
                band_paths[band_name] = matches[0]
            else:
                print(f"  ⚠ {band_name} not found")
                return False
        
        # Open first band for metadata
        ds_first = gdal.Open(str(band_paths['B02']), gdal.GA_ReadOnly)
        x_size, y_size = ds_first.RasterXSize, ds_first.RasterYSize
        projection = ds_first.GetProjection()
        geotransform = ds_first.GetGeoTransform()
        
        # Create output
        driver = gdal.GetDriverByName('GTiff')
        out_ds = driver.Create(
            str(output_path), x_size, y_size, len(bands),
            gdal.GDT_UInt16,
            options=['COMPRESS=LZW', 'TILED=YES', 'BIGTIFF=YES']
        )
        out_ds.SetProjection(projection)
        out_ds.SetGeoTransform(geotransform)
        
        # Write bands
        for i, (band_name, band_path) in enumerate(band_paths.items(), start=1):
            ds_band = gdal.Open(str(band_path), gdal.GA_ReadOnly)
            data = ds_band.GetRasterBand(1).ReadAsArray()
            out_band = out_ds.GetRasterBand(i)
            out_band.WriteArray(data)
            out_band.SetDescription(band_name)
            out_band.FlushCache()
            ds_band = None
        
        out_ds = None
        ds_first = None
        return True
        
    except Exception as e:
        print(f"  ❌ Band stacking failed: {e}")
        return False


def align_corine_to_s2(corine_path, s2_path, output_path):
    """
    Align CORINE raster to match Sentinel-2 tile geometry.
    
    Parameters:
    -----------
    corine_path : Path
        Path to CORINE GeoTIFF
    s2_path : Path
        Path to stacked S2 GeoTIFF (reference)
    output_path : Path
        Output path for aligned CORINE
    
    Returns:
    --------
    bool : Success status
    """
    try:
        # Get S2 geometry
        s2_ds = gdal.Open(str(s2_path), gdal.GA_ReadOnly)
        gt = s2_ds.GetGeoTransform()
        ulx, xres, _, uly, _, yres = gt
        lrx = ulx + (s2_ds.RasterXSize * xres)
        lry = uly + (s2_ds.RasterYSize * yres)
        s2_ds = None
        
        # Use gdalwarp for reliable reprojection
        cmd = [
            'gdalwarp',
            '-s_srs', 'EPSG:3035',
            '-t_srs', 'EPSG:32633',  # UTM 33N
            '-te', str(ulx), str(lry), str(lrx), str(uly),
            '-tr', str(xres), str(abs(yres)),
            '-r', 'near',
            '-co', 'COMPRESS=LZW',
            '-co', 'TILED=YES',
            '-overwrite',
            str(corine_path),
            str(output_path)
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode != 0:
            print(f"  ❌ gdalwarp failed: {result.stderr}")
            return False
        
        return True
        
    except Exception as e:
        print(f"  ❌ CORINE alignment failed: {e}")
        return False


def check_corine_coverage(corine_path):
    """
    Check if aligned CORINE has valid data.
    
    Returns:
    --------
    tuple : (has_valid_data, unique_classes)
    """
    ds = gdal.Open(str(corine_path), gdal.GA_ReadOnly)
    data = ds.GetRasterBand(1).ReadAsArray()
    ds = None
    
    unique = np.unique(data)
    valid_classes = unique[(unique >= 1) & (unique <= 44)]
    
    return len(valid_classes) > 0, valid_classes


print("✓ Processing functions defined")

In [ ]:
def extract_patches(s2_path, corine_path, patch_size=3, stride=None, max_patches=50000):
    """
    Extract training patches from S2 imagery with CORINE labels.
    
    Parameters:
    -----------
    s2_path : Path
        Path to stacked S2 GeoTIFF
    corine_path : Path
        Path to aligned CORINE GeoTIFF
    patch_size : int
        Patch size in pixels
    stride : int
        Stride for extraction (None = patch_size)
    max_patches : int
        Maximum patches to extract
    
    Returns:
    --------
    tuple : (patches, labels, metadata) or (None, None, None)
    """
    try:
        # Open datasets
        s2_ds = gdal.Open(str(s2_path), gdal.GA_ReadOnly)
        corine_ds = gdal.Open(str(corine_path), gdal.GA_ReadOnly)
        
        n_bands = s2_ds.RasterCount
        height = s2_ds.RasterYSize
        width = s2_ds.RasterXSize
        
        if stride is None:
            stride = patch_size
        
        # Process in chunks
        chunk_size = 2000
        patches, labels, coords = [], [], []
        
        for y_chunk in range(0, height, chunk_size):
            for x_chunk in range(0, width, chunk_size):
                y_end = min(y_chunk + chunk_size, height)
                x_end = min(x_chunk + chunk_size, width)
                
                if (y_end - y_chunk) < patch_size or (x_end - x_chunk) < patch_size:
                    continue
                
                # Read chunk
                s2_chunk = np.zeros((y_end - y_chunk, x_end - x_chunk, n_bands), dtype=np.uint16)
                for i in range(n_bands):
                    s2_chunk[:, :, i] = s2_ds.GetRasterBand(i + 1).ReadAsArray(
                        x_chunk, y_chunk, x_end - x_chunk, y_end - y_chunk
                    )
                
                corine_chunk = corine_ds.GetRasterBand(1).ReadAsArray(
                    x_chunk, y_chunk, x_end - x_chunk, y_end - y_chunk
                )
                
                # Extract patches
                for y_local in range(0, y_end - y_chunk - patch_size + 1, stride):
                    for x_local in range(0, x_end - x_chunk - patch_size + 1, stride):
                        patch = s2_chunk[y_local:y_local + patch_size, 
                                        x_local:x_local + patch_size, :]
                        
                        # Get center pixel label
                        center_y = y_local + patch_size // 2
                        center_x = x_local + patch_size // 2
                        label = corine_chunk[center_y, center_x]
                        
                        # Skip invalid labels
                        if label < 1 or label > 44:
                            continue
                        if np.any(patch == 0):  # Missing data
                            continue
                        
                        patches.append(patch)
                        labels.append(label)
                        coords.append((y_chunk + y_local, x_chunk + x_local))
                        
                        if len(patches) >= max_patches:
                            break
                    if len(patches) >= max_patches:
                        break
                if len(patches) >= max_patches:
                    break
            if len(patches) >= max_patches:
                break
        
        s2_ds = None
        corine_ds = None
        
        if len(patches) == 0:
            return None, None, None
        
        # Convert to arrays
        patches = np.array(patches, dtype=np.uint16)
        labels = np.array(labels, dtype=np.uint8)
        
        # Create metadata
        unique_labels, counts = np.unique(labels, return_counts=True)
        metadata = {
            's2_tile': s2_path.stem,
            'corine_file': corine_path.name,
            'patch_size': patch_size,
            'stride': stride,
            'n_patches': len(patches),
            'n_bands': patches.shape[3],
            'n_classes': len(unique_labels),
            'label_distribution': {int(k): int(v) for k, v in zip(unique_labels, counts)},
            'extraction_date': datetime.now().isoformat(),
            'patch_shape': list(patches.shape),
            'bands': ['B02', 'B03', 'B04', 'B08']
        }
        
        return patches, labels, metadata
        
    except Exception as e:
        print(f"  ❌ Patch extraction failed: {e}")
        import traceback
        traceback.print_exc()
        return None, None, None


print("✓ Patch extraction function defined")

## Section 5: Process Tiles

### Main Processing Loop

This cell will process all selected tiles:
1. Stack S2 bands
2. Align CORINE
3. Extract patches
4. Save results

In [ ]:
def process_tile(safe_path, corine_path, output_dir, patch_size, stride, max_patches, skip_existing=True):
    """
    Process a single Sentinel-2 tile.
    
    Returns:
    --------
    dict : Processing results
    """
    tile_name = safe_path.stem
    result = {
        'tile': tile_name,
        'status': 'unknown',
        'n_patches': 0,
        'n_classes': 0,
        'output_file': None
    }
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Check for existing output
    output_npz = output_dir / f"patches_{tile_name}_stacked_data.npz"
    if skip_existing and output_npz.exists():
        print(f"  ⏭ Skipping (output exists)")
        result['status'] = 'skipped'
        result['output_file'] = str(output_npz)
        return result
    
    # Step 1: Stack bands
    print(f"  📦 Stacking S2 bands...")
    s2_stacked = output_dir / f"{tile_name}_stacked.tif"
    if not s2_stacked.exists():
        if not stack_s2_bands(safe_path, s2_stacked):
            result['status'] = 'failed_stacking'
            return result
    
    # Step 2: Align CORINE
    print(f"  🗺️ Aligning CORINE...")
    corine_aligned = output_dir / f"corine_aligned_{tile_name}.tif"
    if not corine_aligned.exists():
        if not align_corine_to_s2(corine_path, s2_stacked, corine_aligned):
            result['status'] = 'failed_alignment'
            return result
    
    # Step 3: Check coverage
    has_coverage, classes = check_corine_coverage(corine_aligned)
    if not has_coverage:
        print(f"  ⚠ No valid CORINE data (tile outside coverage?)")
        result['status'] = 'no_coverage'
        return result
    
    # Step 4: Extract patches
    print(f"  ✂️ Extracting patches...")
    patches, labels, metadata = extract_patches(
        s2_stacked, corine_aligned, 
        patch_size=patch_size, 
        stride=stride, 
        max_patches=max_patches
    )
    
    if patches is None:
        result['status'] = 'no_patches'
        return result
    
    # Step 5: Save results
    print(f"  💾 Saving {len(patches):,} patches...")
    np.savez_compressed(output_npz, patches=patches, labels=labels)
    
    metadata_file = output_dir / f"patches_{tile_name}_stacked_metadata.json"
    with open(metadata_file, 'w') as f:
        json.dump(metadata, f, indent=2)
    
    result['status'] = 'success'
    result['n_patches'] = len(patches)
    result['n_classes'] = metadata['n_classes']
    result['output_file'] = str(output_npz)
    result['metadata'] = metadata
    
    return result


print("✓ Tile processing function defined")

In [ ]:
# ============================================================
# MAIN PROCESSING LOOP
# ============================================================

print("="*70)
print("PROCESSING SENTINEL-2 TILES")
print("="*70)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Tiles to process: {len(available_tiles)}")
print()

# Store results
all_results = []

# Process each tile
for i, tile_path in enumerate(available_tiles, 1):
    print(f"\n[{i}/{len(available_tiles)}] Processing: {tile_path.name}")
    
    result = process_tile(
        safe_path=tile_path,
        corine_path=CORINE_PATH,
        output_dir=OUTPUT_DIR,
        patch_size=PATCH_SIZE,
        stride=STRIDE,
        max_patches=MAX_PATCHES,
        skip_existing=SKIP_EXISTING
    )
    
    all_results.append(result)
    
    if result['status'] == 'success':
        print(f"  ✅ Success: {result['n_patches']:,} patches, {result['n_classes']} classes")
    elif result['status'] == 'skipped':
        print(f"  ⏭ Skipped (already processed)")
    else:
        print(f"  ❌ Status: {result['status']}")

# Summary
print("\n" + "="*70)
print("PROCESSING SUMMARY")
print("="*70)
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

successful = [r for r in all_results if r['status'] == 'success']
skipped = [r for r in all_results if r['status'] == 'skipped']
failed = [r for r in all_results if r['status'] not in ['success', 'skipped']]

print(f"\nResults:")
print(f"  ✅ Successful: {len(successful)}")
print(f"  ⏭ Skipped: {len(skipped)}")
print(f"  ❌ Failed: {len(failed)}")

if successful:
    total_patches = sum(r['n_patches'] for r in successful)
    print(f"\n  Total patches extracted: {total_patches:,}")
    print(f"  Output directory: {OUTPUT_DIR}")

## Section 6: Analyze and Visualize Results

### View Processing Results

In [ ]:
# Display detailed results
print("Detailed Results per Tile:")
print("-" * 80)

for result in all_results:
    status_icon = "✅" if result['status'] == 'success' else "⏭" if result['status'] == 'skipped' else "❌"
    print(f"{status_icon} {result['tile'][:50]}...")
    print(f"    Status: {result['status']}")
    if result['n_patches'] > 0:
        print(f"    Patches: {result['n_patches']:,}")
        print(f"    Classes: {result['n_classes']}")
    print()

### Visualize Sample Patches

In [ ]:
def visualize_patches(patches, labels, n_samples=12, title="Sample Patches"):
    """
    Visualize sample patches with their labels.
    """
    # Get diverse samples
    unique_labels = np.unique(labels)
    
    sample_indices = []
    for label in unique_labels:
        indices = np.where(labels == label)[0]
        n_from_class = min(2, len(indices))  # Up to 2 samples per class
        sample_indices.extend(np.random.choice(indices, n_from_class, replace=False))
        if len(sample_indices) >= n_samples:
            break
    
    sample_indices = sample_indices[:n_samples]
    
    # Create figure
    n_cols = 4
    n_rows = (len(sample_indices) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_rows == 1 and n_cols == 1 else axes
    
    for i, idx in enumerate(sample_indices):
        patch = patches[idx]
        label = labels[idx]
        
        # Create RGB composite
        rgb = patch[:, :, [2, 1, 0]].astype(float)  # B04, B03, B02 = R, G, B
        p2, p98 = np.percentile(rgb, (2, 98))
        rgb = np.clip((rgb - p2) / (p98 - p2 + 1e-6), 0, 1)
        
        # Upscale for visibility
        rgb_upscaled = np.repeat(np.repeat(rgb, 20, axis=0), 20, axis=1)
        
        axes[i].imshow(rgb_upscaled)
        class_name = CORINE_CLASSES.get(label, "Unknown")[:25]
        axes[i].set_title(f"Class {label}: {class_name}", fontsize=10)
        axes[i].axis('off')
    
    # Hide unused
    for i in range(len(sample_indices), len(axes)):
        axes[i].axis('off')
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


# Load and visualize first successful result
successful_results = [r for r in all_results if r['status'] == 'success']

if successful_results:
    result = successful_results[0]
    data = np.load(result['output_file'])
    patches = data['patches']
    labels = data['labels']
    
    print(f"Loaded {len(patches):,} patches from: {Path(result['output_file']).name}")
    visualize_patches(patches, labels, n_samples=12, title=f"Sample Patches from {result['tile'][:40]}...")
else:
    print("No successful results to visualize.")

### Class Distribution Analysis

In [ ]:
def plot_class_distribution(labels, title="Class Distribution"):
    """
    Plot the distribution of CORINE classes.
    """
    unique, counts = np.unique(labels, return_counts=True)
    
    # Sort by count
    sort_idx = np.argsort(counts)[::-1]
    unique = unique[sort_idx]
    counts = counts[sort_idx]
    
    # Get class names
    class_names = [f"{c}: {CORINE_CLASSES.get(c, 'Unknown')[:20]}" for c in unique]
    colors = [CORINE_COLORS.get(c, '#888888') for c in unique]
    
    # Create figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Bar chart
    bars = ax1.barh(range(len(unique)), counts, color=colors)
    ax1.set_yticks(range(len(unique)))
    ax1.set_yticklabels(class_names, fontsize=9)
    ax1.set_xlabel('Number of Patches')
    ax1.set_title('Patch Count by Class')
    ax1.invert_yaxis()
    
    # Add count labels
    for i, (count, bar) in enumerate(zip(counts, bars)):
        ax1.text(count + max(counts)*0.01, i, f'{count:,}', va='center', fontsize=8)
    
    # Pie chart (top 10)
    top_n = min(10, len(unique))
    other_count = counts[top_n:].sum() if len(counts) > top_n else 0
    
    pie_counts = list(counts[:top_n])
    pie_labels = [f"{c}" for c in unique[:top_n]]
    pie_colors = colors[:top_n]
    
    if other_count > 0:
        pie_counts.append(other_count)
        pie_labels.append('Other')
        pie_colors.append('#888888')
    
    ax2.pie(pie_counts, labels=pie_labels, colors=pie_colors, 
            autopct='%1.1f%%', startangle=90)
    ax2.set_title('Top 10 Classes Distribution')
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"\nStatistics:")
    print(f"  Total patches: {len(labels):,}")
    print(f"  Unique classes: {len(unique)}")
    print(f"  Most common: Class {unique[0]} ({CORINE_CLASSES.get(unique[0], 'Unknown')}) - {counts[0]:,} ({counts[0]/len(labels)*100:.1f}%)")
    print(f"  Least common: Class {unique[-1]} ({CORINE_CLASSES.get(unique[-1], 'Unknown')}) - {counts[-1]:,} ({counts[-1]/len(labels)*100:.1f}%)")


# Analyze first successful result
if successful_results:
    result = successful_results[0]
    data = np.load(result['output_file'])
    labels = data['labels']
    
    plot_class_distribution(labels, title=f"Class Distribution: {result['tile'][:40]}...")
else:
    print("No successful results to analyze.")

## Section 7: Combine Multiple Tiles (Optional)

If you processed multiple tiles, you can combine them into a single training dataset.

In [ ]:
def combine_datasets(result_list, output_path):
    """
    Combine patches from multiple tiles into a single dataset.
    
    Parameters:
    -----------
    result_list : list
        List of processing results with output_file paths
    output_path : Path
        Output path for combined dataset
    """
    all_patches = []
    all_labels = []
    tile_sources = []
    
    for result in result_list:
        if result['status'] not in ['success', 'skipped']:
            continue
        if not result['output_file'] or not Path(result['output_file']).exists():
            continue
        
        data = np.load(result['output_file'])
        patches = data['patches']
        labels = data['labels']
        
        all_patches.append(patches)
        all_labels.append(labels)
        tile_sources.extend([result['tile']] * len(labels))
        
        print(f"  Added {len(labels):,} patches from {result['tile'][:40]}...")
    
    if not all_patches:
        print("No data to combine!")
        return None
    
    # Concatenate
    combined_patches = np.concatenate(all_patches, axis=0)
    combined_labels = np.concatenate(all_labels, axis=0)
    
    # Shuffle
    indices = np.random.permutation(len(combined_labels))
    combined_patches = combined_patches[indices]
    combined_labels = combined_labels[indices]
    
    # Save
    np.savez_compressed(output_path, patches=combined_patches, labels=combined_labels)
    
    print(f"\n✓ Combined dataset saved: {output_path}")
    print(f"  Total patches: {len(combined_labels):,}")
    print(f"  Unique classes: {len(np.unique(combined_labels))}")
    print(f"  File size: {Path(output_path).stat().st_size / (1024**2):.1f} MB")
    
    return combined_patches, combined_labels


# Combine all successful results
if len(successful_results) > 1:
    print("Combining datasets from multiple tiles...\n")
    combined_path = Path(OUTPUT_DIR) / "combined_training_data.npz"
    combined_patches, combined_labels = combine_datasets(all_results, combined_path)
    
    if combined_patches is not None:
        plot_class_distribution(combined_labels, title="Combined Dataset Class Distribution")
elif len(successful_results) == 1:
    print("Only one tile processed - no need to combine.")
else:
    print("No successful results to combine.")

## Section 8: Save Visualization (Optional)

In [ ]:
if CREATE_VISUALIZATIONS and successful_results:
    result = successful_results[0]
    data = np.load(result['output_file'])
    patches = data['patches']
    labels = data['labels']
    
    # Create visualization
    unique_labels = np.unique(labels)
    n_samples = min(6, len(unique_labels))
    
    sample_indices = []
    for label in unique_labels[:n_samples]:
        idx = np.where(labels == label)[0][0]
        sample_indices.append(idx)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for i, idx in enumerate(sample_indices):
        patch = patches[idx]
        label = labels[idx]
        
        rgb = patch[:, :, [2, 1, 0]].astype(float)
        p2, p98 = np.percentile(rgb, (2, 98))
        rgb = np.clip((rgb - p2) / (p98 - p2 + 1e-6), 0, 1)
        rgb_upscaled = np.repeat(np.repeat(rgb, 20, axis=0), 20, axis=1)
        
        axes[i].imshow(rgb_upscaled)
        class_name = CORINE_CLASSES.get(label, "Unknown")
        axes[i].set_title(f"Class {label}: {class_name}\n{patch.shape[0]}x{patch.shape[1]} pixels", fontsize=10)
        axes[i].axis('off')
    
    for i in range(len(sample_indices), len(axes)):
        axes[i].axis('off')
    
    plt.suptitle("Sample Patches with CORINE Labels", fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    viz_path = Path(OUTPUT_DIR) / f"visualization_{result['tile']}.png"
    plt.savefig(viz_path, dpi=150, bbox_inches='tight')
    print(f"✓ Visualization saved: {viz_path}")
    plt.show()

## Summary & Next Steps

### What We Accomplished
✅ Processed Sentinel-2 imagery with CORINE Land Cover labels  
✅ Created aligned raster datasets  
✅ Extracted training patches for ML models  
✅ Analyzed class distributions  

### Output Files
- `*_stacked.tif` - Stacked S2 bands (B02, B03, B04, B08)
- `corine_aligned_*.tif` - CORINE aligned to S2 tile
- `patches_*_data.npz` - Training patches and labels
- `patches_*_metadata.json` - Extraction metadata

### Key Takeaways
- CORINE provides consistent land cover labels across Europe
- Proper alignment is critical for accurate labeling
- Class imbalance is common - consider sampling strategies
- Quality control (cloud filtering) improves data quality

### Prepare for Lab 4
Next lab: **Training ML Models with Extracted Patches**
- Load the `.npz` files created here
- Train classification models
- Evaluate on held-out test data

---

**Great work!** Your training data is ready for machine learning! 🚀